In [86]:
import nflreadpy as nfl
import pandas as pd

In [87]:
player_points = nfl.load_ff_opportunity(seasons=[2025])

In [88]:
player_points = player_points.to_pandas()

In [89]:
player_points.head()

,season,posteam,week,game_id,player_id,full_name,position,pass_attempt,rec_attempt,rush_attempt,...,total_yards_gained_diff_team,total_touchdown_team,total_touchdown_exp_team,total_touchdown_diff_team,total_first_down_team,total_first_down_exp_team,total_first_down_diff_team,total_fantasy_points_team,total_fantasy_points_exp_team,total_fantasy_points_diff_team
0,2025,ARI,1.0,2025_01_ARI_NO,00-0035228,Kyler Murray,QB,29.0,0.0,7.0,...,-40.64,2.0,3.17,-1.17,16.0,16.31,-0.31,63.9,75.66,-11.76
1,2025,ARI,1.0,2025_01_ARI_NO,00-0037744,Trey McBride,TE,0.0,9.0,0.0,...,-40.64,2.0,3.17,-1.17,16.0,16.31,-0.31,63.9,75.66,-11.76
2,2025,NO,1.0,2025_01_ARI_NO,00-0039376,Spencer Rattler,QB,46.0,0.0,4.0,...,-81.34,1.0,2.92,-1.92,17.0,23.07,-6.07,65.1,86.84,-21.74
3,2025,NO,1.0,2025_01_ARI_NO,00-0037545,Rashid Shaheed,WR,0.0,9.0,0.0,...,-81.34,1.0,2.92,-1.92,17.0,23.07,-6.07,65.1,86.84,-21.74
4,2025,NO,1.0,2025_01_ARI_NO,None,None,None,0.0,5.0,0.0,...,-81.34,1.0,2.92,-1.92,17.0,23.07,-6.07,65.1,86.84,-21.74


In [90]:
player_points = player_points[player_points['week'] < 19]

In [91]:
player_points.columns.tolist()

['season',
 'posteam',
 'week',
 'game_id',
 'player_id',
 'full_name',
 'position',
 'pass_attempt',
 'rec_attempt',
 'rush_attempt',
 'pass_air_yards',
 'rec_air_yards',
 'pass_completions',
 'receptions',
 'pass_completions_exp',
 'receptions_exp',
 'pass_yards_gained',
 'rec_yards_gained',
 'rush_yards_gained',
 'pass_yards_gained_exp',
 'rec_yards_gained_exp',
 'rush_yards_gained_exp',
 'pass_touchdown',
 'rec_touchdown',
 'rush_touchdown',
 'pass_touchdown_exp',
 'rec_touchdown_exp',
 'rush_touchdown_exp',
 'pass_two_point_conv',
 'rec_two_point_conv',
 'rush_two_point_conv',
 'pass_two_point_conv_exp',
 'rec_two_point_conv_exp',
 'rush_two_point_conv_exp',
 'pass_first_down',
 'rec_first_down',
 'rush_first_down',
 'pass_first_down_exp',
 'rec_first_down_exp',
 'rush_first_down_exp',
 'pass_interception',
 'rec_interception',
 'pass_interception_exp',
 'rec_interception_exp',
 'rec_fumble_lost',
 'rush_fumble_lost',
 'pass_fantasy_points_exp',
 'rec_fantasy_points_exp',
 'rush_f

In [92]:
player_points['total_fantasy_points_exp'].head()

0    22.01
1    15.43
2    22.24
3    14.13
4     7.27
Name: total_fantasy_points_exp, dtype: float64

In [93]:
player_season_proj = player_points.groupby(['player_id', 'full_name', 'position']).agg({'total_fantasy_points_exp': 'sum'}).reset_index()
player_season_proj = player_season_proj.rename(columns={'total_fantasy_points_exp': 'season_fantasy_points_exp'})
player_season_proj.head()

,player_id,full_name,position,season_fantasy_points_exp
0,00-0022942,Philip Rivers,QB,34.00
1,00-0023459,Aaron Rodgers,QB,230.89
2,00-0026158,Joe Flacco,QB,170.59
3,00-0026300,Josh Johnson,QB,30.14
4,00-0026498,Matthew Stafford,QB,348.92


In [94]:
player_season_proj['position_ranking'] = player_season_proj.groupby('position')['season_fantasy_points_exp'].rank(method='min', ascending=False).astype(int)
player_season_proj.head(20)

,player_id,full_name,position,season_fantasy_points_exp,position_ranking
0,00-0022942,Philip Rivers,QB,34.00,51
1,00-0023459,Aaron Rodgers,QB,230.89,16
2,00-0026158,Joe Flacco,QB,170.59,26
3,00-0026300,Josh Johnson,QB,30.14,53
4,00-0026498,Matthew Stafford,QB,348.92,3
5,00-0027973,Andy Dalton,QB,15.75,61
6,00-0028118,Tyrod Taylor,QB,69.67,43
7,00-0029263,Russell Wilson,QB,74.41,42
8,00-0029604,Kirk Cousins,QB,114.93,36
9,00-0029892,Kyle Juszczyk,RB,52.60,71


In [95]:
def replacement_points(df, pos, rank):
    x = df[(df["position"] == pos) & (df["position_ranking"] == rank)]
    if x.empty:
        raise ValueError(f"No {pos} at position_ranking={rank}")
    return float(x["season_fantasy_points_exp"].iloc[0])

rep_qb = replacement_points(player_season_proj, "QB", 14)
rep_rb = replacement_points(player_season_proj, "RB", 28)
rep_wr = replacement_points(player_season_proj, "WR", 34)
rep_te = replacement_points(player_season_proj, "TE", 14)

In [96]:
def raw_par(df):
    if df["position"].iloc[0] == "QB":
        x = df.copy()
        x['raw_par'] = x['season_fantasy_points_exp'] - rep_qb
        return x
    elif df["position"].iloc[0] == "RB":
        x = df.copy()
        x['raw_par'] = x['season_fantasy_points_exp'] - rep_rb
        return x
    elif df["position"].iloc[0] == "WR":
        x = df.copy()
        x['raw_par'] = x['season_fantasy_points_exp'] - rep_wr
        return x
    elif df["position"].iloc[0] == "TE":
        x = df.copy()
        x['raw_par'] = x['season_fantasy_points_exp'] - rep_te
        return x
player_season_proj = player_season_proj.groupby('position').apply(raw_par).reset_index(drop=True)


C:\Users\atuls\AppData\Local\Temp\ipykernel_448\1156016264.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  player_season_proj = player_season_proj.groupby('position').apply(raw_par).reset_index(drop=True)


In [97]:
player_season_proj.head(20)

,player_id,full_name,position,season_fantasy_points_exp,position_ranking,raw_par
0,00-0022942,Philip Rivers,QB,34.00,51,-213.05
1,00-0023459,Aaron Rodgers,QB,230.89,16,-16.16
2,00-0026158,Joe Flacco,QB,170.59,26,-76.46
3,00-0026300,Josh Johnson,QB,30.14,53,-216.91
4,00-0026498,Matthew Stafford,QB,348.92,3,101.87
5,00-0027973,Andy Dalton,QB,15.75,61,-231.30
6,00-0028118,Tyrod Taylor,QB,69.67,43,-177.38
7,00-0029263,Russell Wilson,QB,74.41,42,-172.64
8,00-0029604,Kirk Cousins,QB,114.93,36,-132.12
9,00-0030565,Geno Smith,QB,195.60,23,-51.45


In [98]:
def compute_utility(raw_par: float, alpha: float, min_utility: float) -> float:
    utility = raw_par if raw_par >= 0 else raw_par * alpha
    return utility - min_utility

In [99]:
ALPHA_BY_POS = {
    "QB": 0.08,   
    "RB": 0.20,
    "WR": 0.25,
    "TE": 0.15
}

In [100]:
player_season_proj["alpha"] = player_season_proj["position"].map(ALPHA_BY_POS)

player_season_proj["utility_raw"] = player_season_proj.apply(
    lambda r: r["raw_par"] if r["raw_par"] >= 0 else r["raw_par"] * r["alpha"],
    axis=1
)

In [101]:
min_utility = player_season_proj["utility_raw"].min()
player_season_proj["utility"] = player_season_proj["utility_raw"] - min_utility

In [102]:
player_season_proj.head(20)

,player_id,full_name,position,season_fantasy_points_exp,position_ranking,raw_par,alpha,utility_raw,utility
0,00-0022942,Philip Rivers,QB,34.00,51,-213.05,0.08,-17.0440,25.3085
1,00-0023459,Aaron Rodgers,QB,230.89,16,-16.16,0.08,-1.2928,41.0597
2,00-0026158,Joe Flacco,QB,170.59,26,-76.46,0.08,-6.1168,36.2357
3,00-0026300,Josh Johnson,QB,30.14,53,-216.91,0.08,-17.3528,24.9997
4,00-0026498,Matthew Stafford,QB,348.92,3,101.87,0.08,101.8700,144.2225
5,00-0027973,Andy Dalton,QB,15.75,61,-231.30,0.08,-18.5040,23.8485
6,00-0028118,Tyrod Taylor,QB,69.67,43,-177.38,0.08,-14.1904,28.1621
7,00-0029263,Russell Wilson,QB,74.41,42,-172.64,0.08,-13.8112,28.5413
8,00-0029604,Kirk Cousins,QB,114.93,36,-132.12,0.08,-10.5696,31.7829
9,00-0030565,Geno Smith,QB,195.60,23,-51.45,0.08,-4.1160,38.2365


In [103]:
player_season_proj = player_season_proj[['player_id', 'full_name', 'utility', 'position_ranking']]

In [104]:
player_season_proj.to_csv('player_points.csv', index=False)